# Processing Proteomic data of ProCan-DepMapSanger
Here I perform pre-processing of the proteomics data from the ProCan-DepMapSanger project (https://pmc.ncbi.nlm.nih.gov/articles/PMC9387775/#da0010). The pridepy package was used to query data from the PRIDE database.

About the project:
* PRIDE identifier: PXD030304
* 949 human cancer cell lines, 40 cancer types
* Has 290 overlapping cell lines with the CCLE study
* 187 cell lines from lung
* DIA-MS, six replicates across 6 MS machines, preprocessed using DIA-NN version 1.8
* Also include data of drug response

In [1]:
import os
import anndata as ad
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import umap
import yaml

In [2]:
with open("../../config/config.local.yaml", "r") as f:
    config = yaml.safe_load(f)
basedir = config['code_dir']

data_input_dir = config["cell_line_data_dir"]
cell_line_data_filename = config["cell_line_data_filename"]
cell_line_metadata_filename = config["cell_line_metadata_filename"]

data_output_dir = os.path.join(config["output_data_dir"], "processed_proteomics_data")
if not os.path.exists(data_output_dir):
    os.makedirs(data_output_dir)

## Importing the data

In [3]:
proteome_df = pd.read_csv(os.path.join(data_input_dir, cell_line_data_filename), sep="\t", index_col=0, decimal=",")
metadata_df = pd.read_csv(os.path.join(data_input_dir, cell_line_metadata_filename), sep="\t", index_col=0, decimal=",")
proteome_df.head()

,P37108;SRP14_HUMAN,Q96JP5;ZFP91_HUMAN,Q9Y4H2;IRS2_HUMAN,P36578;RL4_HUMAN,Q6SPF0;SAMD1_HUMAN,O76031;CLPX_HUMAN,Q8WUQ7;CATIN_HUMAN,A6NIH7;U119B_HUMAN,Q9BTD8;RBM42_HUMAN,Q9P258;RCC2_HUMAN,...,P33151;CADH5_HUMAN,Q5EBL4;RIPL1_HUMAN,P49715;CEBPA_HUMAN,Q5TA45;INT11_HUMAN,O14924;RGS12_HUMAN,Q7Z3B1;NEGR1_HUMAN,O60669;MOT2_HUMAN,Q13571;LAPM5_HUMAN,Q96JM2;ZN462_HUMAN,P35558;PCKGC_HUMAN
Project_Identifier,,,,,,,,,,,,,,,,,,,,,
SIDM00018;K052,7.10955,3.41494,NaN,7.86661,3.89547,4.19666,NaN,NaN,3.19088,7.35806,...,NaN,NaN,3.90064,2.63998,NaN,NaN,NaN,NaN,NaN,NaN
SIDM00023;TE-12,6.82802,4.14346,2.23781,7.62878,3.19811,4.60902,NaN,2.47059,3.69535,5.70790,...,NaN,NaN,NaN,3.19608,NaN,NaN,NaN,NaN,NaN,NaN
SIDM00040;TMK-1,7.01426,4.19987,2.44055,8.12459,NaN,4.76881,NaN,NaN,NaN,5.52283,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
SIDM00041;STS-0421,5.28591,3.35789,NaN,7.97268,NaN,4.52092,NaN,NaN,2.73088,4.29429,...,NaN,NaN,NaN,2.79023,NaN,NaN,NaN,NaN,NaN,NaN
SIDM00042;PL4,5.70786,NaN,NaN,6.22574,NaN,4.49579,NaN,NaN,2.87981,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [4]:
metadata_df.head()

,Project_Identifier,Cell_line,Source,Identifier,Gender,Tissue_type,Cancer_type,Cancer_subtype,Haem_lineage,BROAD_ID,...,F6,F7,F8,F9,F10,F11,F12,F13,F14,F15
model_id,,,,,,,,,,,,,,,,,,,,,
SIDM00896,SIDM00896;BC-1,BC-1,ATCC,CVCL_1079,Male,Haematopoietic and Lymphoid,B-Cell Non-Hodgkin's Lymphoma,Primary effusion lymphoma,B lymphoid,ACH-002214,...,-0.460410592,0.769344541,1.655735696,2.109066796,-0.056366659,-0.550704882,0.22918636,0.48678876,0.299904549,-0.319624837
SIDM00312,SIDM00312;L-363,L-363,DSMZ,CVCL_1357,Female,Haematopoietic and Lymphoid,Plasma Cell Myeloma,NaN,B lymphoid,ACH-000183,...,0.586819719,1.267371741,3.302839254,1.927399012,1.67645876,-1.009455463,-0.329970134,0.530467659,0.585515536,-0.092114513
SIDM00277,SIDM00277;EoL-1-cell,EoL-1-cell,RIKEN,CVCL_0258,Male,Haematopoietic and Lymphoid,Acute Myeloid Leukemia,Chronic Eosinophilic Leukemia,Myeloid,ACH-000198,...,-1.023188467,-3.082193494,-1.778517568,0.218588171,-0.191836004,-1.5289751,0.444940467,-0.23824369,1.183521049,-0.245174026
SIDM01119,SIDM01119;NCI-H727,NCI-H727,ATCC,CVCL_1584,Female,Lung,Carcinoid Tumour,Lung Carcinoid Tumor,NaN,ACH-000775,...,1.15562679,1.266913309,-0.128102385,0.1088428,-0.895634273,-0.813593711,0.352701873,-1.68028983,0.4363571,-0.034756331
SIDM00657,SIDM00657;MV-4-11,MV-4-11,ATCC,CVCL_0064,Male,Haematopoietic and Lymphoid,Acute Myeloid Leukemia,Acute Monocytic leukemia,Myeloid,ACH-000045,...,-0.160455371,-2.825198424,-2.446800068,0.334810381,-2.487931171,-1.485627055,0.177513259,-0.218151504,0.501118696,-0.044598198


In [5]:
proteome_df = proteome_df.transpose()
proteome_df.columns = [s.split(";")[1] for s in proteome_df.columns]
proteome_df["UniprotID"] = [s.split(";")[0] for s in proteome_df.index]
proteome_df["Gene"] = [s.split(";")[1] for s in proteome_df.index]
proteome_df["organism"] = [s.split("_")[1] for s in proteome_df["Gene"]]
proteome_df["Gene"] = [s.split("_")[0] for s in proteome_df["Gene"]]
proteome_df.set_index("Gene", inplace=True)
proteome_df.head()

,K052,TE-12,TMK-1,STS-0421,PL4,PCI-4B,PCI-30,HSC-39,H3255,EMC-BAC-2,...,BE-13,MC-IXC,Ramos-2G6-4C10,CGTH-W-1,H9,GR-ST,YMB-1-E,MM1S,UniprotID,organism
Gene,,,,,,,,,,,,,,,,,,,,,
SRP14,7.10955,6.82802,7.01426,5.28591,5.70786,6.73965,6.04591,6.20582,6.53547,6.80469,...,6.66480,7.02345,6.99319,6.31631,6.23087,7.00407,6.76532,6.91982,P37108,HUMAN
ZFP91,3.41494,4.14346,4.19987,3.35789,NaN,4.50199,3.69356,2.88118,NaN,2.78737,...,3.60415,3.12879,3.85570,4.86933,2.71686,NaN,3.75777,3.70284,Q96JP5,HUMAN
IRS2,NaN,2.23781,2.44055,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,3.07386,NaN,NaN,NaN,NaN,NaN,NaN,Q9Y4H2,HUMAN
RL4,7.86661,7.62878,8.12459,7.97268,6.22574,7.47364,7.07092,8.25336,6.47861,7.58653,...,6.99447,7.78652,7.32346,7.62433,7.23503,7.58150,7.24133,6.78319,P36578,HUMAN
SAMD1,3.89547,3.19811,NaN,NaN,NaN,NaN,3.49594,3.35439,NaN,2.14980,...,3.34052,3.24627,3.91503,3.92689,3.42065,NaN,3.05477,1.89549,Q6SPF0,HUMAN


In [6]:
proteome_df.index.name = "Gene"
proteome_df.reset_index(inplace=True)

In [7]:
# load functions for processing data and plotting
from proteome_2_anndata import proteome_2_anndata
from proteome_anndata_plot import *

In [8]:
adata = proteome_2_anndata(proteome_df = proteome_df, 
                           metadata_df = metadata_df,
                           match_sample_column="Cell_line",
                           match_protein_column="Gene",
                           protein_annotation_columns=["UniprotID", "organism"],
                           need_log2=False)
adata

INFO: No log2 intensity threshold specified. Threshold layers not computed.


INFO: AnnData object successfully created and returned.


AnnData object with n_obs × n_vars = 949 × 6692
    obs: 'index', 'Project_Identifier', 'Source', 'Identifier', 'Gender', 'Tissue_type', 'Cancer_type', 'Cancer_subtype', 'Haem_lineage', 'BROAD_ID', 'CCLE_ID', 'ploidy', 'mutational_burden', 'msi_status', 'growth_properties', 'growth', 'size', 'media', 'replicates_correlation', 'number_of_proteins', 'EMT', 'Proteasome', 'TranslationInitiation', 'CopyNumberInstability', 'GeneExpressionCorrelation', 'CopyNumberAttenuation', 'F1', 'F2', 'F3', 'F4', 'F5', 'F6', 'F7', 'F8', 'F9', 'F10', 'F11', 'F12', 'F13', 'F14', 'F15'
    var: 'UniprotID', 'organism'
    uns: 'intensity_threshold'
    obsm: 'number_detection_sample', 'missing_rate_sample'
    varm: 'missing_rate_gene', 'median_intensity', 'sum_intensity'

In [9]:
plot_qc(adata)

<Axes: xlabel='Proteins, ordered by sum of intensity', ylabel='Log2 intensity'>

In [10]:
boxplot_sample_intensities(adata)

<Axes: title={'center': 'Boxplot of intensities per sample (layer: None)'}, xlabel='Samples', ylabel='Intensity'>

In [11]:
plot_sample_completeness(adata)

<Axes: title={'center': 'Sample completeness (layer: None)'}, xlabel='Samples', ylabel='Completeness (fraction non-missing)'>

In [12]:
adata.to_df()

Gene,1433B,1433E,1433F,1433G,1433S,1433T,1433Z,2A5D,2A5E,2A5G,...,ZO2,ZO3,ZPI,ZPLD1,ZPR1,ZRAB2,ZW10,ZWILC,ZWINT,ZYX
Cell_line,,,,,,,,,,,,,,,,,,,,,
BC-1,7.03601,8.00759,5.29918,8.06047,9.00286,7.73755,10.18420,2.64450,1.61465,1.63534,...,2.65595,NaN,NaN,NaN,4.20748,4.61742,2.45898,2.12305,2.44594,NaN
L-363,7.21637,6.99908,4.76455,8.05752,NaN,6.55054,9.39175,2.94373,2.71268,NaN,...,1.95277,NaN,NaN,NaN,3.98919,4.39367,2.43581,2.13892,1.65612,NaN
EoL-1-cell,7.79587,8.04782,5.81756,7.61262,NaN,7.04231,9.69640,3.43704,1.81748,0.75706,...,NaN,NaN,NaN,NaN,4.52350,4.14481,2.78943,1.50250,0.71931,7.92373
NCI-H727,5.21437,5.06841,2.30314,4.42794,6.89186,4.07034,6.80954,NaN,NaN,0.91372,...,2.82575,NaN,NaN,NaN,-1.04043,2.39105,3.24981,NaN,NaN,1.16095
MV-4-11,7.48790,7.37133,5.65960,7.76590,NaN,6.24001,8.83293,3.52667,NaN,2.03021,...,NaN,NaN,NaN,NaN,4.00379,4.01387,2.94725,2.82090,NaN,6.39953
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
NCI-H1304,6.47536,7.30501,5.53525,7.98859,9.89288,6.82812,9.03692,3.78325,3.49713,2.83776,...,1.82589,NaN,3.08459,NaN,3.93498,5.29737,2.75988,2.61092,2.50434,2.03411
SC-1,6.87320,7.76908,5.69065,8.28049,NaN,7.08927,9.29892,2.92155,2.53242,2.56400,...,2.15732,NaN,NaN,NaN,4.05009,5.06646,2.95427,2.62113,NaN,NaN
SCC-25,4.80541,5.82832,3.56780,6.02552,8.59059,4.52708,7.80129,NaN,NaN,NaN,...,4.49126,NaN,NaN,NaN,NaN,3.51867,2.87796,NaN,NaN,NaN


In [13]:
# save the anndata
adata.write(os.path.join(data_output_dir, "PanCancer_Goncalves2022_cancer_cell_lines.h5ad"))

... storing 'Source' as categorical


... storing 'Gender' as categorical


... storing 'Tissue_type' as categorical


... storing 'Cancer_type' as categorical


... storing 'Cancer_subtype' as categorical


...

... storing 'replicates_correlation' as categorical


... storing 'CopyNumberInstability' as categorical


... storing 'GeneExpressionCorrelation' as categorical


... storing 'CopyNumberAttenuation' as categorical


... storing 'organism' as categorical
